# PINN-SFR-Transient — guided walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppigazzini/pinn-sfr-transient/blob/main/notebooks/01_ulof_walkthrough.ipynb)

An **Unprotected Loss of Flow (ULOF)** transient in a sodium-cooled fast reactor:
six-group point kinetics coupled to lumped thermal-hydraulics, closed by
reactivity feedback including the **positive sodium void coefficient**.

This notebook runs the physics end-to-end (numpy/scipy only for §1–6), then trains
the same PINN in **PyTorch (§7)** and **JAX (§8)**. Background:
[`docs/physics_theory.md`](../docs/physics_theory.md),
[`docs/neural_network.md`](../docs/neural_network.md),
[`docs/usage.md`](../docs/usage.md).

> Tip: *Run All*. The first cell installs the package automatically on Google
> Colab. §1–6 need only numpy/scipy; §7 (PyTorch) and §8 (JAX) each train the PINN
> and skip cleanly if that backend is missing. **For a fast run, pick a GPU
> runtime** (*Runtime → Change runtime type → GPU*).

In [ ]:
# --- setup: make `pinn_sfr_transient` importable whether the package is
#     installed (`uv sync`), run from a repo checkout, or on Google Colab ---
import importlib, pathlib, subprocess, sys

_REPO = "git+https://github.com/ppigazzini/pinn-sfr-transient.git"


def _ensure_package():
    try:
        import pinn_sfr_transient  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    # repo checkout? put src/ on the path
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "src" / "pinn_sfr_transient").exists():
            sys.path.insert(0, str(cand / "src"))
            importlib.invalidate_caches()
            return
    # otherwise (e.g. Google Colab): install from GitHub. Colab already ships
    # numpy/scipy/matplotlib, so on Colab use --no-deps -- upgrading them inside
    # a live kernel corrupts numpy's ABI ("cannot import name '_center'"). If you
    # hit that error, numpy was upgraded earlier this session: restart the runtime
    # (Runtime > Restart session) and run again.
    cmd = [sys.executable, "-m", "pip", "install"]
    if "google.colab" in sys.modules:
        cmd.append("--no-deps")
    cmd.append(_REPO)
    print(f"Installing pinn-sfr-transient from {_REPO} ...")
    subprocess.run(cmd, check=True)
    importlib.invalidate_caches()


_ensure_package()

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pinn_sfr_transient
from pinn_sfr_transient import SFRParams, solve_reference
from pinn_sfr_transient.physics import reactivity, void_fraction, flow_fraction

print("pinn_sfr_transient", pinn_sfr_transient.__version__)

## 1. The model and its parameters

`SFRParams` holds every physical parameter. Derived constants (`UA`, `W0`,
`Cf`, `Cc`, the rescaled `beta_i`, and the criticality offset) are recomputed
automatically so the nominal steady state is exact.

In [ ]:
p = SFRParams()
print(f"beta_eff = {p.beta_eff:.4f}   Lambda = {p.Lambda:.1e} s   (stiff!)")
print(f"alpha_f (Doppler) = {p.alpha_f:+.1e} /K   <- negative, prompt, stabilising")
print(f"alpha_void        = {p.alpha_void:+.1e}    <- POSITIVE: the SFR signature")
print(f"T_onset = {p.T_onset} K (demonstration hot-channel threshold)")
print()
print("delayed groups:")
for i, (lam, beta_i) in enumerate(zip(p.lambda_i, p.beta_i), 1):
    print(f"  group {i}:  lambda={lam:7.4f} 1/s   beta_i={beta_i:.5f}")
y0 = p.steady_state()
print(f"\nnominal steady precursors C_i0 (note the 1e4-1e5 magnitudes): "
      f"{np.array2string(y0[1:7], precision=0)}")

## 2. Run the stiff reference simulation

`solve_reference` integrates the 9-state system from the nominal steady state
with an implicit Radau scheme. This is the held-out "ground truth" used only for
final PINN validation.

In [ ]:
traj = solve_reference(p)
i_peak = int(np.argmax(traj.P))
print(f"peak power  P_max = {traj.P.max():.3f}  at t = {traj.t[i_peak]:.2f} s")
print(f"final power P_end = {traj.P[-1]:.3f}")
print(f"peak T_f          = {traj.Tf.max():.1f} K")
print(f"peak T_c          = {traj.Tc.max():.1f} K")
print(f"peak void         = {void_fraction(traj.Tc.max(), p):.3f}")

## 3. Visualise the transient

Four panels: power, temperatures (with the void onset), sodium void fraction,
and total reactivity (in dollars) alongside the flow coast-down.

In [ ]:
t = traj.t
void = void_fraction(traj.Tc, p)
rho_dollars = reactivity(traj.Tf, traj.Tc, p) / p.beta_eff
g = flow_fraction(t, p)

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0,0].plot(t, traj.P, color="#b22222", lw=2); ax[0,0].axhline(1, color="grey", ls=":")
ax[0,0].set(title="Normalised power P(t)", xlabel="t [s]", ylabel="P / P_nom")
ax[0,1].plot(t, traj.Tf, color="#d35400", lw=2, label="T_f (fuel)")
ax[0,1].plot(t, traj.Tc, color="#2980b9", lw=2, label="T_c (coolant)")
ax[0,1].axhline(p.T_onset, color="green", ls="--", label="void onset")
ax[0,1].set(title="Temperatures", xlabel="t [s]", ylabel="T [K]"); ax[0,1].legend()
ax[1,0].plot(t, void, color="#8e44ad", lw=2); ax[1,0].set_ylim(-0.02, 1.02)
ax[1,0].set(title="Sodium void fraction", xlabel="t [s]", ylabel="void [-]")
ax[1,1].plot(t, rho_dollars, color="#16a085", lw=2, label="total rho [$]")
ax[1,1].plot(t, g, color="#7f8c8d", ls="--", label="flow g(t)")
ax[1,1].axhline(0, color="grey", ls=":")
ax[1,1].set(title="Reactivity & flow", xlabel="t [s]"); ax[1,1].legend()
fig.suptitle("ULOF reference transient", fontweight="bold")
fig.tight_layout(); plt.show()

## 4. The feedback competition

The bounded excursion is the result of a race: the **positive void** feedback
pushes power up as the coolant boils, while the **negative Doppler** feedback
grows as the fuel heats. Decomposing the total reactivity into its components
shows exactly how Doppler overtakes void and turns the transient over.

In [ ]:
doppler = p.alpha_f * (traj.Tf - p.Tf0) / p.beta_eff
coolant = p.alpha_c * (traj.Tc - p.Tc0) / p.beta_eff
voidrho = p.alpha_void * void_fraction(traj.Tc, p) / p.beta_eff
total   = doppler + coolant + voidrho + p.rho_ext / p.beta_eff

plt.figure(figsize=(9, 5))
plt.plot(t, voidrho, label="void (+)", color="#8e44ad", lw=2)
plt.plot(t, doppler, label="Doppler (-)", color="#d35400", lw=2)
plt.plot(t, coolant, label="coolant (-)", color="#2980b9", lw=2)
plt.plot(t, total,   label="total", color="black", lw=2.5)
plt.axhline(0, color="grey", ls=":")
plt.title("Reactivity components [$]"); plt.xlabel("t [s]"); plt.ylabel("rho [$]")
plt.legend(); plt.tight_layout(); plt.show()

## 5. Verify: the normalized residuals equal the physical ODEs

The PINN trains on *normalized* residuals. Here we confirm they vanish on the
reference trajectory — the same check as `tests/test_consistency.py` [3], and the
reason the PINN math is trustworthy even before any training.

In [ ]:
y = traj.y
dydt = np.gradient(y, t, axis=1)
P, Tf, Tc = y[0], y[7], y[8]
C = y[1:7]
C_i0 = (p.beta_i / (p.Lambda * p.lambda_i))[:, None]
c = C / C_i0
rho = reactivity(Tf, Tc, p); gg = flow_fraction(t, p)

R_p = p.Lambda*dydt[0] - ((rho - p.beta_eff)*P + (p.beta_i[:,None]*c).sum(0))
R_c = dydt[1:7]/C_i0 - p.lambda_i[:,None]*(P[None,:] - c)
R_tf = dydt[7] - ((p.P0/p.Cf)*P - (p.UA/p.Cf)*(Tf - Tc))
R_tc = dydt[8] - ((p.UA/p.Cc)*(Tf - Tc) - (p.W0*gg/p.Cc)*(Tc - p.Tin))
s = slice(5, -5)
print(f"max |R_p|  = {np.abs(R_p[s]).max():.1e}   (machine-precision; FD-limited elsewhere)")
print(f"max |R_c|  = {np.abs(R_c[:,s]).max():.1e}")
print(f"max |R_tf| = {np.abs(R_tf[s]).max():.1e}")
print(f"max |R_tc| = {np.abs(R_tc[s]).max():.1e}")

## 6. Parameter exploration — strength of the void coefficient

Sweep the sodium void coefficient. A larger positive `alpha_void` injects more
reactivity once boiling starts, raising the peak — until Doppler still wins and
bounds the transient. This is the kind of study the PINN is meant to learn as a
*family* (see the operator-learning extension in `docs/neural_network.md` §8).

In [ ]:
plt.figure(figsize=(9, 5))
for av in [4e-3, 6e-3, 8e-3, 1.0e-2]:
    pv = SFRParams(alpha_void=av)
    tv = solve_reference(pv)
    plt.plot(tv.t, tv.P, lw=2, label=f"alpha_void = {av:.0e}  (peak {tv.P.max():.2f})")
plt.axhline(1, color="grey", ls=":")
plt.title("Effect of the sodium void coefficient on the power excursion")
plt.xlabel("t [s]"); plt.ylabel("P / P_nom"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Train the PINN — PyTorch backend

Train the physics-informed network on the ODE residuals only (no data) and
validate against the reference from §2. PyTorch here is object-oriented / eager;
§8 is the *same* PINN in JAX ([`docs/neural_network.md`](../docs/neural_network.md)
§9 compares the two — they are equally first-class). (`jacobian="forward"` uses
`torch.func`; it auto-falls-back to reverse mode on builds where forward-mode
misbehaves.)

> **A GPU runtime trains ~5× faster — use one** (*Runtime → Change runtime type →
> GPU*). §7 and §8 use the **identical** optimisation budget, so it's a fair
> comparison. On a Colab **T4** both backends fit well and train ~5× faster than
> on CPU — about a minute, versus several on CPU (the exact time varies a lot by
> instance). The cell prints its wall-clock time. Use a **GPU, not a TPU**: this problem needs
> float64, which torch can't run on a TPU runtime (and JAX falls back to CPU there).


In [ ]:
try:
    import time

    import torch  # noqa: F401

    from pinn_sfr_transient.pinn_torch import TrainConfig, predict, relative_l2, train

    # Use the GPU when present: on a Colab T4 this trains ~5x faster than CPU and
    # converges well (float64 is throttled on consumer GPUs but still beats CPU
    # here). Falls back to CPU otherwise, incl. TPU runtimes (torch can't use a TPU).
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # identical optimisation budget to the JAX cell (§8)
    cfg = TrainConfig(adam_iters=3000, lbfgs_iters=600, n_colloc=4000, device=device)
    print(f"PyTorch backend: {cfg.device}")
    print("Training the PyTorch PINN...")
    t0 = time.perf_counter()
    model = train(p, cfg)
    print(f"PyTorch training took {time.perf_counter() - t0:.0f} s")
    pinn = predict(model, n=800)
    errs = relative_l2(pinn, {"t": traj.t, "P": traj.P, "Tf": traj.Tf, "Tc": traj.Tc})
    print("relative L2 vs reference:", {k: f"{v:.2e}" for k, v in errs.items()})

    plt.figure(figsize=(9, 4))
    plt.plot(traj.t, traj.P, color="#b22222", lw=2, label="reference")
    plt.plot(pinn["t"], pinn["P"], "--", color="#1f3a93", lw=2, label="PINN")
    plt.title("Power: reference vs PINN")
    plt.xlabel("t [s]")
    plt.ylabel("P / P_nom")
    plt.legend()
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print("PyTorch not installed. Run:  uv sync --extra torch-cpu")
    print("then re-run this cell for a live PINN demo.")

## 8. The same PINN in JAX

The identical PINN in JAX (Equinox + Optax) instead of PyTorch — same residuals
and recipe, in a *functional* idiom: the Equinox model is an immutable PyTree and
Optax applies gradients as a pure transform. On Colab `jax` is pre-installed; this
cell adds `equinox` and `optax`. Guarded: trains if `jax` is installed, else
prints how to install it.


In [ ]:
try:
    import os
    import time

    # float64 (required for this stiff problem) is unsupported on TPU; on a TPU
    # runtime run JAX on CPU instead. Must be set before JAX initialises.
    if any(k in os.environ for k in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_WORKER_ID")):
        os.environ["JAX_PLATFORMS"] = "cpu"

    import jax  # noqa: F401

    try:
        import equinox  # noqa: F401
        import optax  # noqa: F401
    except ModuleNotFoundError:
        import importlib
        import subprocess
        import sys

        print("installing equinox + optax (jax itself is pre-installed on Colab)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "equinox", "optax"], check=True)
        importlib.invalidate_caches()

    if jax.default_backend() == "tpu":
        print(
            "JAX is on a TPU, which doesn't support the float64 this PINN needs.\n"
            "Switch to a GPU runtime (Runtime > Change runtime type > GPU), then\n"
            "Runtime > Restart session, and re-run."
        )
    else:
        from pinn_sfr_transient.pinn_jax import TrainConfig, predict, relative_l2, train

        print(f"JAX backend: {jax.default_backend()}")
        # identical optimisation budget to the PyTorch cell (§7)
        cfg = TrainConfig(adam_iters=3000, lbfgs_iters=600, n_colloc=4000)
        print("Training the JAX PINN...")
        t0 = time.perf_counter()
        model = train(p, cfg)
        print(f"JAX training took {time.perf_counter() - t0:.0f} s")
        pinn = predict(model, p, n=800)
        errs = relative_l2(pinn, {"t": traj.t, "P": traj.P, "Tf": traj.Tf, "Tc": traj.Tc})
        print("relative L2 vs reference:", {k: f"{v:.2e}" for k, v in errs.items()})

        plt.figure(figsize=(9, 4))
        plt.plot(traj.t, traj.P, color="#b22222", lw=2, label="reference")
        plt.plot(pinn["t"], pinn["P"], "--", color="#1f3a93", lw=2, label="PINN")
        plt.title("Power: reference vs PINN")
        plt.xlabel("t [s]")
        plt.ylabel("P / P_nom")
        plt.legend()
        plt.tight_layout()
        plt.show()
except ModuleNotFoundError:
    print("JAX not installed. Run:  uv sync --extra jax-cpu")
    print("then re-run this cell for a live PINN demo.")

## Next steps

* Use a **GPU runtime** and re-run §7/§8 — both backends train ~5× faster on a
  Colab T4 (about a minute, vs several on CPU).
* Raise `adam_iters`/`lbfgs_iters` (the `TrainConfig` defaults are 15000/600)
  for a sub-percent PINN fit.
* Compare the PyTorch (§7) and JAX (§8) backends — same residuals, two idioms;
  see [`docs/neural_network.md`](../docs/neural_network.md) §9.
* Swap in physical sodium-saturation parameters (raise `T_onset`, shorten
  `tau_pump`).
* See [`docs/usage.md`](../docs/usage.md) for the full CLI, library API, and
  troubleshooting; [`docs/references.md`](../docs/references.md) for the
  bibliography.
